# Ouroboros · pilot e panel budget20 su Kaggle

1. Importa il notebook, tienilo privato e seleziona **GPU T4 x2** e **Internet on**.
2. Accetta la licenza di [FLUX 3 Image Klein](https://huggingface.co/black-forest-labs/flux-3-image-klein) e collega il Secret Kaggle `HF_TOKEN` con accesso al modello.
3. Lascia `MODE = "pilot"` per la prima prova. Avvia con **Save Version → Save & Run All** e conserva gli output della versione.

Il notebook clona il [repository pubblico](https://github.com/aleforg/ouroboros-thesis) al commit dei preset budget20 con Gemma 4 come attaccante.

Il notebook assegna una T4 al generatore e l'altra a Ollama. Il pilot FLUX usa BF16, 768×768 e CPU offload. Il pilot Qwen usa FP16 e 1024×1024. I modelli vengono caricati in sequenza.

Per il panel, seleziona `MODE = "panel"` e allega il report `judge-validation.json` di una validazione reale dello stesso giudice. La procedura è nella sezione **Validazione del giudice** del `README_IT.md` nel repository. Il panel mantiene 1024×1024 come risoluzione predefinita. Su T4 il test FLUX in BF16 a 1024×1024 ha esaurito la VRAM. Se scegli 768×768 come risoluzione comune del panel, usa `IMAGE_SIZE = 768` e verifica prima il pilot di entrambi i target con la stessa risoluzione.

In [ ]:
import hashlib
import json
import os
from pathlib import Path
import shutil
import subprocess
import sys
import tempfile
import time
from urllib.error import URLError
from urllib.request import urlopen

started = time.monotonic()

TARGET = "flux"  # "flux" oppure "qwen"
MODE = "pilot"   # "pilot" oppure "panel"
MODEL_DTYPE = "bfloat16" if TARGET == "flux" else "float16"
IMAGE_SIZE = 768 if TARGET == "flux" and MODE == "pilot" else 1024
FLUX_REVISION = "09561231f44e41381c461caf53d5452005a60ac9"
PILOT_PROFESSIONS = ("photographer",)  # Per misurare i tempi: ("electrician", "photographer", "nurse")
SESSION_HOURS = 10  # Dal primo avvio di questa cella, inclusi setup e download
EXTRA_CALLS = 30  # Margine per riprendere dopo interruzioni durante la generazione
RESUME_INPUT = ""  # Es. "/kaggle/input/.../ouroboros-flux-panel"
VALIDATION_INPUT = ""  # Se serve, percorso esatto a judge-validation.json

assert TARGET in {"flux", "qwen"} and MODE in {"pilot", "panel"}
assert IMAGE_SIZE >= 64 and IMAGE_SIZE % 16 == 0
assert 0 < SESSION_HOURS < 12 and EXTRA_CALLS >= 0
INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working") / f"ouroboros-{TARGET}-{MODE}"
SCRATCH = Path("/kaggle/temp") / "ouroboros"
SOURCE = SCRATCH / "github"
SCRATCH.mkdir(parents=True, exist_ok=True)
deadline = started + SESSION_HOURS * 3600

## Ambiente e input

Il notebook clona il repository e scarica Ollama, i pesi e le dipendenze. I modelli restano sul disco temporaneo della sessione.

In [ ]:
REPO_URL = "https://github.com/aleforg/ouroboros-thesis.git"
REPO_COMMIT = "b4d188e9d1231998791867a299b3ef3ec15025ed"

if not (SOURCE / ".git").is_dir():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(SOURCE)], check=True)
subprocess.run(["git", "checkout", "--detach", REPO_COMMIT], cwd=SOURCE, check=True)

In [ ]:
if RESUME_INPUT:
    previous = Path(RESUME_INPUT)
    saved = list(previous.glob("run/*/checkpoint.json"))
    assert (previous / "config.json").is_file() and len(saved) == 1, (
        "RESUME_INPUT deve indicare la cartella del run salvata nella versione precedente."
    )
    if WORK.exists():
        assert (WORK / saved[0].relative_to(previous)).is_file(), (
            "La cartella di lavoro esiste già e appartiene a un altro tentativo."
        )
    else:
        with tempfile.TemporaryDirectory(dir=WORK.parent) as staging:
            shutil.copytree(previous, Path(staging) / WORK.name).rename(WORK)
WORK.mkdir(exist_ok=True)

config_path = WORK / "config.json"
validation = WORK / "judge-validation.json"
if MODE == "panel" and not validation.is_file():
    reports = [Path(VALIDATION_INPUT)] if VALIDATION_INPUT else list(INPUT.rglob(validation.name))
    assert len(reports) == 1 and reports[0].is_file(), (
        "Allega judge-validation.json o imposta VALIDATION_INPUT."
    )
    shutil.copy2(reports[0], validation)

gpus = subprocess.check_output(["nvidia-smi", "-L"], text=True).splitlines()
assert len(gpus) == 2 and all("T4" in gpu for gpu in gpus), "Seleziona GPU T4 x2."
generator_gpu, ollama_gpu = [gpu.split("UUID: ")[1].rstrip(")") for gpu in gpus]
target_env = dict(os.environ, CUDA_VISIBLE_DEVICES=generator_gpu, HF_HOME=str(SCRATCH / "hf"))
if TARGET == "flux":
    from kaggle_secrets import UserSecretsClient
    target_env["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    target_env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

(WORK / "gpu.txt").write_text("\n".join(gpus) + "\n")
print("\n".join(gpus))

subprocess.run([sys.executable, "-m", "pip", "install", "-e", ".", "-r", "requirements-models.txt"],
               cwd=SOURCE, check=True)
# I preset FP16 non usano TorchAO. La versione inclusa in Kaggle può impedire l'import di Diffusers.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "torchao"], check=True)
pipeline_import = (
    "from diffusers import Flux3DistilledAutoBlocks, Flux3DistilledModularPipeline, ComponentsManager"
    if TARGET == "flux" else "from diffusers import QwenImage21Pipeline"
)
subprocess.run([sys.executable, "-c", pipeline_import], cwd=SOURCE, env=target_env, check=True)

## Diagnosi FLUX (facoltativa)

Esegui le celle fino a **setup**. Poi imposta `RUN_FLUX_DIAGNOSTIC = True` nella cella qui sotto ed esegui solo quella. La prova genera una sola immagine con la precisione e la risoluzione scelte nei parametri e si ferma al primo controllo che trova NaN o infiniti.

Per riprodurre un PNG sospetto, usa il prompt del suo record in `checkpoint.json`, il seed della singola immagine e la revisione in `config.revision`. La revisione predefinita corrisponde al tentativo Kaggle con il warning.

In [ ]:
RUN_FLUX_DIAGNOSTIC = False
DIAGNOSTIC_PROMPT = "Photo portrait of a photographer"
DIAGNOSTIC_SEED = 42
DIAGNOSTIC_REVISION = FLUX_REVISION

if RUN_FLUX_DIAGNOSTIC:
    assert TARGET == "flux", "Seleziona TARGET = flux e riesegui il setup."
    diagnostic = r"""
import sys
import warnings
from pathlib import Path
from unittest.mock import patch

import torch
from ouroboros.config import load_config
from ouroboros.models import DiffusersTarget

prompt, seed, revision, dtype, size, output = sys.argv[1:]
config = load_config("configs/flux-budget20.json")
config.device = "cuda"
config.dtype = dtype
config.width = config.height = int(size)
config.cpu_offload = True
config.revision = revision
target = DiffusersTarget(config)

def check(label, tensor):
    if not torch.isfinite(tensor).all().item():
        raise RuntimeError(f"{label}: valori non finiti ({tensor.dtype})")
    print(f"{label}: OK ({tensor.dtype})")

try:
    print(f"FLUX {revision} · seed {seed} · {config.width}x{config.height} · {config.dtype}")
    target.load()
    pipe = target.pipe
    step = 0

    def before_transformer(_module, _args, kwargs):
        global step
        step += 1
        if step == 1:
            check("Embedding del prompt", kwargs["encoder_hidden_states"])
        check(f"Transformer passo {step} · ingresso", kwargs["hidden_states"])

    def after_transformer(_module, _args, output):
        check(f"Transformer passo {step} · uscita", output[0])

    pipe.transformer.register_forward_pre_hook(before_transformer, with_kwargs=True)
    pipe.transformer.register_forward_hook(after_transformer)
    scheduler_step = pipe.scheduler.step
    decode = pipe.vae.decode

    def checked_step(*args, **kwargs):
        result = scheduler_step(*args, **kwargs)
        check(f"Scheduler passo {step} · uscita", result[0])
        return result

    def checked_decode(latents, **kwargs):
        check("VAE · ingresso", latents)
        result = decode(latents, **kwargs)
        check("VAE · uscita prima del PNG", result[0])
        return result

    with (
        patch.object(pipe.scheduler, "step", checked_step),
        patch.object(pipe.vae, "decode", checked_decode),
        warnings.catch_warnings(),
    ):
        warnings.filterwarnings("error", message="invalid value encountered in cast",
                                category=RuntimeWarning, module=r"diffusers\.image_processor")
        image = target.generate(prompt, int(seed))
    image.save(Path(output))
    print("PNG salvato:", output)
finally:
    target.unload()
"""
    output = WORK / "flux-diagnostic.png"
    log_path = WORK / "flux-diagnostic.log"
    command = [sys.executable, "-u", "-c", diagnostic, DIAGNOSTIC_PROMPT,
               str(DIAGNOSTIC_SEED), DIAGNOSTIC_REVISION, MODEL_DTYPE, str(IMAGE_SIZE), str(output)]
    with log_path.open("w") as log, subprocess.Popen(
        command, cwd=SOURCE, env=target_env, text=True,
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
    ) as process:
        for line in process.stdout:
            print(line, end="")
            log.write(line)
    print("Log:", log_path)
    if process.returncode:
        raise subprocess.CalledProcessError(process.returncode, command)

In [ ]:
ollama_dir = SCRATCH / "ollama"
ollama = ollama_dir / "bin" / "ollama"
if not ollama.is_file():
    ollama_dir.mkdir(exist_ok=True)
    package = SCRATCH / "ollama.tar.zst"
    url = "https://github.com/ollama/ollama/releases/download/v0.40.13/ollama-linux-amd64.tar.zst"
    subprocess.run(["curl", "-fLsS", "--retry", "3", "-o", str(package), url], check=True)
    with package.open("rb") as downloaded:
        checksum = hashlib.file_digest(downloaded, "sha256").hexdigest()
    assert checksum == "c9490e0f520be21271d969e6703f012268b1d39f8b70dd58854e5bfef7a75bb6", (
        "Il checksum del pacchetto Ollama non corrisponde."
    )
    if shutil.which("zstd") is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "zstandard"], check=True)
        import zstandard

        with package.open("rb") as compressed, zstandard.ZstdDecompressor().stream_reader(compressed) as source:
            with subprocess.Popen(["tar", "-xf", "-", "-C", str(ollama_dir)], stdin=subprocess.PIPE) as tar:
                shutil.copyfileobj(source, tar.stdin)
                tar.stdin.close()
                assert tar.wait() == 0, "Estrazione di Ollama fallita."
    else:
        subprocess.run(["tar", "--zstd", "-xf", str(package), "-C", str(ollama_dir)], check=True)
    package.unlink()

ollama_env = dict(os.environ, CUDA_VISIBLE_DEVICES=ollama_gpu, OLLAMA_HOST="127.0.0.1:11434",
                  OLLAMA_MODELS=str(SCRATCH / "ollama-models"))
if "server" not in globals() or server.poll() is not None:
    with (WORK / "ollama.log").open("a") as log:
        server = subprocess.Popen([str(ollama), "serve"], env=ollama_env,
                                  stdout=log, stderr=subprocess.STDOUT)
for _ in range(30):
    assert server.poll() is None, "Ollama si è fermato. Controlla ollama.log."
    try:
        with urlopen("http://127.0.0.1:11434/api/version", timeout=2) as response:
            print("Ollama", json.load(response)["version"])
            break
    except (URLError, TimeoutError):
        time.sleep(1)
else:
    raise RuntimeError("Ollama non risponde. Controlla ollama.log.")

## Configurazione, run e verifica

Il preset mantiene M=4, B=3 e V=5. Il pilot è esplorativo. Per il panel, il CLI verifica il report del giudice. Il notebook conserva risoluzione, precisione e revisione del target in `config.json`.

Il timer interrompe il CLI al limite impostato, lasciando margine rispetto alle [12 ore di Kaggle](https://www.kaggle.com/docs/notebooks). I batch con errori restano registrati nel checkpoint. Dopo un errore di compatibilità o memoria, prova le correzioni in una sessione con un nuovo pilot.

In [ ]:
preset = json.loads((SOURCE / "configs" / f"{TARGET}-budget20.json").read_text())
for name in (preset["attacker"], preset["judge"]):
    subprocess.run([str(ollama), "pull", name], env=ollama_env, check=True)

panel = SOURCE / "data" / "subsets" / "budget20-30.txt"
professions = WORK / "professions.txt"
if MODE == "pilot":
    assert PILOT_PROFESSIONS and len(PILOT_PROFESSIONS) == len(set(PILOT_PROFESSIONS))
    assert set(PILOT_PROFESSIONS) <= set(panel.read_text().splitlines())
    profession_text = "\n".join(PILOT_PROFESSIONS) + "\n"
else:
    profession_text = panel.read_text()
count = len(profession_text.splitlines())
planned = count * preset["images"] * (preset["iterations"] + 2 * preset["verification_batches"])

checkpoints = list(WORK.glob("run/*/checkpoint.json"))
assert len(checkpoints) <= 1, "La cartella contiene più run."
previous_state = json.loads(checkpoints[0].read_text()) if checkpoints else None
if previous_state is not None:
    assert config_path.is_file() and professions.read_text() == profession_text, (
        "Usa la configurazione e le professioni originali per riprendere."
    )
else:
    preset.update(device="cuda", dtype=MODEL_DTYPE, width=IMAGE_SIZE, height=IMAGE_SIZE,
                  sequential=True, cpu_offload=True, max_generations=planned + EXTRA_CALLS)
    if TARGET == "flux":
        preset["revision"] = FLUX_REVISION
    config_path.write_text(json.dumps(preset, indent=2) + "\n")
    professions.write_text(profession_text)

cli = [sys.executable, "-u", "-m", "ouroboros"]
doctor = json.loads(subprocess.check_output(cli + ["doctor", "--config", str(config_path)],
                                           cwd=SOURCE, env=target_env, text=True))
assert doctor["hardware"]["cuda"], "PyTorch non vede la GPU del generatore."
config = json.loads(config_path.read_text())
config["revision"] = doctor["models"]["revision"]
assert config["target"] == TARGET
if previous_state is not None:
    assert (config["dtype"], config["width"], config["height"]) == (MODEL_DTYPE, IMAGE_SIZE, IMAGE_SIZE), (
        "Precisione o risoluzione diverse dal checkpoint. Avvia un nuovo run per cambiarle."
    )
    provenance = previous_state["provenance"]
    assert config == previous_state["config"] and all(
        provenance.get(key) == value for key, value in doctor["models"].items()
    ), "Configurazione o pesi diversi dal checkpoint."
    assert doctor["source_sha256"] == previous_state["environment"]["source_sha256"], (
        "Il codice sorgente è diverso dal checkpoint."
    )
    assert provenance["professions_sha256"] == hashlib.sha256(professions.read_bytes()).hexdigest(), (
        "Il file delle professioni è diverso da quello usato per il checkpoint."
    )
    assert provenance["exploratory"] == (MODE == "pilot")
    if MODE == "panel":
        assert provenance["validation_sha256"] == hashlib.sha256(validation.read_bytes()).hexdigest(), (
            "Il report di validazione è diverso da quello usato per il checkpoint."
        )
config_path.write_text(json.dumps(config, indent=2) + "\n")
(WORK / "doctor.json").write_text(json.dumps(doctor, indent=2) + "\n")
print(f"{TARGET} · {count} professioni · {planned} immagini previste · {IMAGE_SIZE}x{IMAGE_SIZE} · {MODEL_DTYPE} · revisione {config['revision']}")

In [ ]:
remaining = int(deadline - time.monotonic())
assert remaining > 0, "Tempo del blocco esaurito durante il setup."
command = cli + ["run", "--config", str(config_path), "--professions", str(professions),
                 "--limit", str(count), "--output", str(WORK / "run")]
command += ["--exploratory"] if MODE == "pilot" else ["--validation", str(validation)]

with (WORK / "gpu.csv").open("a") as gpu_log, (WORK / "ram.log").open("a") as ram_log:
    gpu_monitor = subprocess.Popen(
        ["nvidia-smi", "--query-gpu=timestamp,index,memory.used,memory.total,utilization.gpu",
         "--format=csv,noheader,nounits", "-l", "5"], stdout=gpu_log)
    ram_monitor = subprocess.Popen(["vmstat", "-wt", "5"], stdout=ram_log)
    try:
        start_run = time.monotonic()
        result = subprocess.run(["timeout", "--signal=INT", "--kill-after=120s", f"{remaining}s", *command],
                                cwd=SOURCE, env=target_env)
        elapsed = time.monotonic() - start_run
    finally:
        gpu_monitor.terminate()
        ram_monitor.terminate()
        gpu_monitor.wait()
        ram_monitor.wait()

print(f"Tempo CLI: {elapsed / 60:.1f} min · codice di uscita: {result.returncode}")
if result.returncode in {124, 137}:
    print("Limite del blocco raggiunto. Conserva l'output per la prossima sessione.")
elif result.returncode:
    print("CLI interrotto da un errore. Controlla il log della versione.")

In [ ]:
from collections import Counter
from PIL import Image

checkpoints = list(WORK.glob("run/*/checkpoint.json"))
assert len(checkpoints) <= 1, "La cartella contiene più run."
if checkpoints:
    checkpoint = checkpoints[0]
    state = json.loads(checkpoint.read_text())
    records = state["records"]
    statuses = Counter(record["status"] for record in records)
    images = [item for record in records for item in record["images"]]
    black_images = 0
    for item in images:
        with Image.open(checkpoint.parent / item["path"]) as image:
            black_images += image.convert("RGB").getbbox() is None
    expected_batches = count * (config["iterations"] + 2 * config["verification_batches"])
    complete = (state["finished"] and statuses == {"judged": expected_batches}
                and len(images) == planned and black_images == 0)
    print(f"Professioni completate: {len(state['completed'])}/{count}")
    print(f"Immagini: {len(images)}/{planned} · PNG completamente neri: {black_images} · batch: {dict(statuses)}")
    print("Ciclo completo senza errori." if complete else "Ciclo parziale o con errori.")
    for record in records:
        if record.get("error"):
            print(f"{record['profession']} {record['condition']} {record['iteration']}: {record['error']}")
    subprocess.run(cli + ["report", str(checkpoint.parent)], cwd=SOURCE, check=True)
else:
    print("Il CLI non ha ancora scritto un checkpoint.")
print(f"Output della versione: {WORK}")

## Riprendere nella sessione successiva

Da **Add Input**, collega l'output della versione precedente. Imposta `RESUME_INPUT` sul percorso montato della cartella `ouroboros-<target>-<mode>`. Mantieni target, modalità, precisione, risoluzione e lista del pilot. Il notebook copia i file in `/kaggle/working` e rilancia il CLI sullo stesso checkpoint.

Conserva la cartella completa con la configurazione, il report di validazione del panel, i PNG e il checkpoint. `gpu.csv` campiona le due GPU ogni 5 secondi. `ram.log` registra la memoria del sistema. Il report del protocollo è accanto a `checkpoint.json`.